In [1]:
#need to change all code to 2d
from random import seed
from random import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math
from mpl_toolkits.mplot3d import Axes3D
%matplotlib inline
import csv
from matplotlib.colors import LinearSegmentedColormap
from scipy.optimize import curve_fit
def fit_func(x, a, b):
    return a*x + b

import csv

def csv_to_array(file_path):
    data_array = []
    with open(file_path, 'r') as file:
        csv_reader = csv.reader(file)
        for row in csv_reader:
            data_array.append(row)
    data_array=np.array(data_array)
    return data_array

def csv_to_numpy_array(file_path):
    data_array = np.loadtxt(file_path, delimiter=',', dtype=str)
    return data_array

In [2]:
def nuactivefluid_2d_kn(g0=1,ngnotinput=0,k0=0.1,nknotinput=1,cc=1,ca=1,\
                        d=1,gammainput=1,hill=1,uinput=0,boxsize=200, finestep=1,dtv=0.001,checkp=20):
    #nutrient dependent growth
    #parameter
    g=g0 #maximum growth rate
    c0=cc #initial nutrient density at each site
    cap=ca #maximum cell/spore density at each site
    #care=cares# cell density rescale according to unit size and finner step
    k=k0 #maximum sporulation rate
    D=d #diffusivity of nutrient
    z=hill #hill coefficient
    dt=dtv #timestep
    ngnot=ngnotinput #growth rate sensitivity to nutrient
    nknot=nknotinput #sporulation rate sensitivity to nutrient
    gamma=gammainput #nutrient conversion rate to cell
    u=uinput # used to plot continuous graph for different gamma
    lengthscale=int(boxsize*finestep) #proposed length of growing space
    finner=finestep
    finarea=finestep*finestep
    finint=int(finestep)

    print("nutrient density=",c0,",cell capacity=",cap,",ng=",ngnot,",nk=",nknot,",gamma=",gamma)
    
    #checkpoint for looping
    checkpoint=checkp #could be time...
    
    #varying properties
    cap=ca*60*60/(finner*finner) # rescale density with unit size=60um
    phi=np.zeros(lengthscale) #cell percent
    spore=np.zeros(lengthscale) #spore percent
    nutrient=np.zeros(lengthscale)+c0 #local nutrient
    space=np.linspace(0,lengthscale,lengthscale)
    
    i=0
    areaapprox=np.zeros(lengthscale)
    while i<lengthscale:
        areaapprox[i]=(2*i+1)/(2*finarea)
        i=i+1

    #radius
    rovertime=[] #
    rsimu=[] #
    rsimutip=[] #
    #rsporesimutip=[]
    
    sumrecorded=[] #total amount of cells, could/may be used to analytical derive radius
        
    #time
    timerecorded=[]
    timenow=0

    #flatness of nutrient
    #gradnatR=[]
    #gradnmax=[]
    #nsum=[]
    
    
    #cellpercent
    cellpercentrecorded=[]
    #edgespeed 
    edgespeedrecorded=[]
    
    #profile recording
    cellovertime=[]
    sporeovertime=[]
    nutrientovertime=[]
    defineedge=1/10# 0.1unit =0.1*60um #np.ceil(1000)
    cap=ca*60*60/(finner*finner) # rescale density with unit size=60um
    gamma=gammainput/cap #rescale gamma with unit 
    #print(defineedge)

    #initial condition
    i=0
    restofcell=cap*((defineedge)**2)*finner*finner
    restofcellr=cap*((defineedge)**2)*finner*finner
    while i<defineedge*finner:
        if i<np.floor(np.sqrt(restofcellr/cap)):
            phi[i]=cap
            restofcell=restofcell-cap*(2*i+1)
        else:
            phi[i]=restofcell/(2*i+1)
            restofcell=0
        spore[i]=0
        i=i+1
    print(phi[0:finestep+1])
    #print(i-1)
    
    #print(sum(np.array(phi)>0))
    R=finner*(2*(sum(phi*areaapprox)+sum(spore*areaapprox))/(cap))**0.5 #outer radius based on initial condition
    edge=np.floor(R)
    nutrientthresh=sum(nutrient*areaapprox*(phi>0))/(sum(areaapprox))
    #print("nth=",nutrientthresh)
    #print(R)

    effgrowth=g*(nutrient)/(ngnot)#(nutrient*0+ngnot)
    effsporulation=k*1/((nutrient/nknot)**z+1)#k*(np.array(nutrient)<=0)
    #effsporulation=k*(np.array(nutrient)<=0.1)#k*nknot/(nutrient+nknot)
    #plt.figure(u)
    #plt.plot(space/finner,phi/max(phi),"r-",label="cell percent")
    #plt.plot(space/finner,spore/(max(spore)),"g-",label="spore density")
    #plt.plot(space/finner,nutrient/(max(nutrient)),"p-",label="nutrient")
    #plt.xlim(0,lengthscale/finner)
    #plt.legend()
    #plt.show()
    #u=u+1
        
    i=1
    rv=np.zeros(lengthscale)
    
    while i<=edge:
        rv[i]=rv[i-1]+phi[i-1]*effgrowth[i-1]*((2*i-1)/(2*finarea))/cap
        i=i+1
        
    graphtimec=np.linspace(0,checkpoint+1,checkpoint*2)#[0,1,2,5,30,checkpoint+1] 
    gtc=0
    profilerecordtime=0
    #looping
    while checkpoint>0:
        gradientphi=np.zeros(lengthscale)
        fluxphi=np.zeros(lengthscale)
        fluxspore=np.zeros(lengthscale)
        nutrientpp=np.zeros(lengthscale)
        rovertime.append(R)
        rsimu.append(np.argmax(phi))

        loc=int(edge)
        sumrv=sum(effgrowth*phi*areaapprox)/cap
        #cellpercent=(sum(phi*areaapprox))/(sum(phi*areaapprox)+sum(spore*areaapprox))
        sumv=(rv[loc]+effgrowth[loc]*phi[loc]*((2*loc+1)/(2*finarea))/cap)/(R/finner)
        #print(sum(spore),sum(effsporulation))
        #radius recording
        i=len(phi)-1
        while i>0:
            if phi[i]+spore[i]<=0:
                i=i-1
            else:
                break
        rsimutip.append(i/finner)
        #i=len(spore)-1
        #while i>0:
        #    if spore[i]<cap*0.5:
        #        i=i-1
        #    else:
        #        break
        #rsporesimutip.append(i/finner)
        timerecorded.append(timenow)
        #cellpercentrecorded.append(cellpercent)
        sumrecorded.append(sum(phi*areaapprox)+sum(spore*areaapprox))
        edgespeedrecorded.append(sumv)

        #flatness of nutrient
        i=0
        nutrientp=[]
        while i<len(nutrient)-1:
            nutrientp.append((nutrient[i+1]-nutrient[i])*finner)
            i=i+1
            
        #gradnatR.append((nutrient[loc+1]-nutrient[loc])*finner)
        #gradnmax.append(np.max(nutrientp))
        #nsum.append(sum(nutrient*areaapprox))
        
        
        #if timenow>=profilerecordtime:
        #    profilerecordtime=profilerecordtime+dt*1000
        #    cellovertime.append(phi)
        #    sporeovertime.append(spore)
        #    nutrientovertime.append(nutrient)

        #finding gradient/flux
        i=0
        while i<len(phi):
            if i==0:
                fluxphi[i]=phi[i]*rv[i+1]/((2*i+1)/(2*finarea))
                #fluxspore[i]=spore[i]*rv[i+1]/((2*i+1)/(2*finarea))
                nutrientpp[i]=(nutrient[i+1]-nutrient[i])*((i+1))/((2*i+1)/(2*finarea))
            elif i==len(phi)-1:
                fluxphi[i]=(phi[i]*rv[i]-phi[i-1]*rv[i])/((2*i+1)/(2*finarea))
                #fluxspore[i]=(spore[i]*rv[i]-spore[i-1]*rv[i])/((2*i+1)/(2*finarea))
                nutrientpp[i]=(nutrient[i-1]-nutrient[i])*(i)/((2*i+1)/(2*finarea))
            else:
                fluxphi[i]=(phi[i]*rv[i+1]-phi[i-1]*rv[i])/((2*i+1)/(2*finarea))
                #fluxspore[i]=(spore[i]*rv[i+1]-spore[i-1]*rv[i])/((2*i+1)/(2*finarea))
                nutrientpp[i]=((nutrient[i+1]-nutrient[i])*((i+1))-(nutrient[i]-nutrient[i-1])*(i))\
                /((2*i+1)/(2*finarea))
            i=i+1
        phinew=phi+dt*(phi*effgrowth-effsporulation*phi-fluxphi)
        sporenew=spore+dt*(effsporulation*phi-fluxspore)
        nutrientnew=nutrient+(D*nutrientpp-phi*effgrowth*gamma)*dt
        Rnew=R+(sumv*dt)*finner
        R=Rnew
        timenow=timenow+dt
        
        #if edge>3:
        #    print(nutrient[int(edge-3):int(edge+3)])
        if np.any(nutrientnew<0)==True:
            print("nutrient concentration below 0--error")
            if edge>3:
                print(nutrient[int(0):int(10)])
                print(nutrientpp[int(0):int(10)])
                print(nutrientnew[int(0):int(10)])
            checkpoint=0
        #print(u)
        if timenow>=graphtimec[gtc]:
            print(graphtimec[gtc])
            gtc=gtc+1
            #plt.figure(u)
            #pltedge=[]
            #pltphi=[]
            #pltspore=[]
            #pltnutrient=[]
            
            #i=len(phi)-1
            #while i>0:
            #    if phi[i]+spore[i]<=0:
            #        i=i-1
            #    else:
            #        break
            #j=i
            #ni=0
            #while j>=0:
            #    pltedge.append(ni/fnn)
            #    pltphi.append(phi[j])
            #    pltspore.append(spore[j])
            #    pltnutrient.append(nutrient[j])
            #    j=j-1
            #    ni=ni+1
            
            #plt.title("profile looking at distance from edge")
            #plt.plot(pltedge,pltphi,"r-",label="cell percent")
            #plt.plot(pltedge,pltspore,"g-",label="spore density")
            #plt.plot(pltedge,pltnutrient,"b-",label="nutrient")
            #plt.xlim(0,min(1000,max(pltedge)))


            #plt.plot(space,pressure,"c-",label="pressure")
            #plt.plot(space,speed,"y-",label="velocity")
            #plt.axvline(x = R, color = 'b', label = 'Biofilm edge')
            #plt.axvline(x = np.argmax(phinew), color = 'm', label = 'simulated Biofilm edge')
            #plt.legend()
            #plt.show()

            #u=u+1
        phi=phinew
        spore=sporenew
        nutrient=nutrientnew
        edge=np.floor(R)
        effgrowth=g*(nutrient)/(ngnot)#(nutrient*0+ngnot)
        effsporulation=0#k*1/((nutrient/nknot)**z+1)#nknot/((nutrient)+nknot)#k*(np.array(nutrient)<=0)
        #effsporulation=k*(np.array(nutrient)<=0.1)#k*nknot/(nutrient+nknot)
        #print(k,sum(effsporulation))
        if edge<lengthscale:
            i=1
            rv=np.zeros(lengthscale)
            while i<=edge:
                rv[i]=rv[i-1]+phi[i-1]*effgrowth[i-1]*((2*i-1)/(2*finarea))/cap
                #print(rv)
                i=i+1
        else:
            checkpoint=0
            
        checkpoint=checkpoint-dt
        if checkpoint<=0:
            print("simulation ends -- time limit reached -- redo needed!")
        if phi[len(phi)-200]+spore[len(phi)-200]>0:
            checkpoint=0
            print("simulation ends -- sufficiently close to edge")
        if nutrient[-1]<nutrientthresh:
            checkpoint=0
            print("simulation ends -- nutrients density sufficiently low")
        #if sumv<max(edgespeedrecorded)*2/3:
        #    checkpoint=0
        #    print("simulation ends -- edgespeed sufficiently low")
        
            
    #plt.figure(u)
    #calculation for speed
    #i=0
    #speedovertime=[]
    #speedott=[]
    #tovert=[]
    
    #while i+1<len(rsimu):
        #speedovertime.append((rsimu[i+1]-rsimu[i])/(timerecorded[i+1]-timerecorded[i]))
    #    tovert.append(timerecorded[i])
    #    i=i+1
    #i=0
    #while i+1<len(rsimutip):
     #   speedott.append((rsimutip[i+1]-rsimutip[i])/(timerecorded[i+1]-timerecorded[i]))
      #  i=i+1
        
    #plt.plot(timerecorded,np.array(rsimutip),"m-",label="simulation")
    avrou=sum(phi)/R
    #rcor=cap/cap-(c0/cap)*(2/(c0-cap*gamma))
    #print(rcor,rcor**0.5,((1/(c0-cap*gamma))*np.exp(np.array([300])*(g/ngnot)*(c0-cap*gamma))+rcor)**0.5)
    anatip=1#((1/(c0-cap*gamma))*(c0*2/cap)*np.exp(np.array(timerecorded)*(g/ngnot)*(c0-cap*gamma))+rcor)**0.5#(((g)/(g-k))**(0.5))*np.exp(np.array(timerecorded)*(g-k)/2)
            #np.exp(np.array(timerecorded)*(sum(effgrowth)/len(effgrowth))*avrou/2)*1
    #anatip=(1/(cap)+(c0/(c0-cap*gamma))*(1/(cap))*(-1+np.exp(np.array(timerecorded)*(g/ngnot)*(c0-cap*gamma))))**0.5
    #plt.plot(timerecorded,sumrecorded,"b-",label="amount of biomass")
    #plt.plot(timerecorded,rovertime,"b-",label="estimated")
    #plt.plot(timerecorded,np.array(anatip),"b-",label="analytic estimated")
    #plt.legend()
    #plt.xlabel("time")
    #plt.ylabel("radius--length of biofilm")
    #plt.savefig("radius temporal growth "+str("g=1,k=0.3,time= ")+str(u)+".pdf")
    #plt.show()
    #u=u+1
    #speedrecorded
    #plt.plot(timerecorded,rovertime,"b-",label="estimated")
    #plt.figure(u)
    #plt.plot(timerecorded,np.array(rsimutip),"m-",label="simulation")
    #plt.plot(timerecorded,sumrecorded,"b-",label="amount of biomass")
    #plt.plot(timerecorded,rovertime,"b-",label="estimated")
    #plt.legend()
    #plt.xlabel("time")
    #plt.ylabel("radius--length of biofilm")
    #plt.show()
    #u=u+1
    #plt.figure(u)
    #plt.plot(tovert,speedovertime)
    #u=u+1
    #plt.figure(u)
    #plt.plot(space/finner,phi,"r-",label="cell density")
    #plt.plot(space/finner,spore/(max(spore)),"g-",label="spore density")
    #plt.plot(space/finner,nutrient/(max(nutrient)),"p-",label="nutrient")
    #plt.xlim(0,lengthscale/finner)
    #plt.legend()
    #plt.show()
    #u=u+1
    plt.figure(u)
    plt.title("edgespeed")
    #plt.plot(space/finner,phi,"r-",label="cell density")
    #plt.plot(space/finner,spore/(max(spore)),"g-",label="spore density")
    plt.plot(timerecorded,edgespeedrecorded,"m-")
    #plt.xlim(0,lengthscale/finner)
    #plt.legend()
    plt.show()
    u=u+1
    plt.figure(u)
    plt.title("radius")
    #plt.plot(space/finner,phi,"r-",label="cell density")
    #plt.plot(space/finner,spore/(max(spore)),"g-",label="spore density")
    plt.plot(timerecorded,rovertime,"r-")#,label="nutrient")
    #plt.xlim(0,lengthscale/finner)
    #plt.legend()
    plt.show()
    u=u+1
    maxspeed=max(edgespeedrecorded)
    rsporesimutip=0
    gradnatR=0
    gradnmax=0
    nsum=0
    
    
    return phi,spore,nutrient,timerecorded,\
cellpercentrecorded,edgespeedrecorded,sumrecorded,rsimu,rsimutip,\
rsporesimutip,rovertime,anatip,maxspeed,u,\
cellovertime,sporeovertime,nutrientovertime,\
gradnatR,gradnmax,nsum
        

In [3]:
gamval=[0.2,0.4,0.6,0.8,1,1.2,1.4,1.6,1.8,2,2.2,2.4,2.6,2.8,3]
ccvval=[0.2,0.4,0.6,0.8,1,1.2,1.4,1.6,1.8,2,2.2,2.4,2.6,2.8,3]
checkpv=40
fnn=10
D=100
dt=0.000001
#recorded=

In [ ]:
i=4#len(ccvval)-1
u=0
#ccv=1.4
gam=3
while i<5:#len(ccvval):#len(gamval):
    #gam=gamval[i]
    ccv=ccvval[i]
    phi,spore,nutrient,timerecorded,cellpercentrecorded,edgespeedrecorded,\
            sumrecorded,\
            rsimu,rsimutip,rsporesimutip,rovert,anati,\
            maxspeed,u,\
            cellovertime,sporeovertime,nutrientovertime,\
            gradnatR,gradnmax,nsum=\
            nuactivefluid_2d_kn(g0=1,ngnotinput=1,k0=0,nknotinput=0.4,\
                                       cc=ccv,ca=1,d=D,gammainput=gam,\
                                hill=1,uinput=u,boxsize=100,finestep=fnn,dtv=dt,checkp=checkpv) 
    name1="rsimutip"+",inutrientcon="+str(ccv)+",gamma="+str(gam)+".csv"
    #np.savetxt(name1, rsimutip, delimiter=",", fmt="%.6f")
    name1="timerecorded"+",inutrientcon="+str(ccv)+",gamma="+str(gam)+".csv"
    #np.savetxt(name1, timerecorded, delimiter=",", fmt="%.6f")
    name1="edgespeedrecorded"+",inutrientcon="+str(ccv)+",gamma="+str(gam)+".csv"
    #np.savetxt(name1, edgespeedrecorded, delimiter=",", fmt="%.6f")
    
    fitlen=-20#-1*int(0.1/dt)#int(len(timerecorded)/(4/3))
    params = curve_fit(fit_func, timerecorded[fitlen:], edgespeedrecorded[fitlen:])
    [h, j] = params[0]
    
    
    if h>0:
        print(name1+" is superlinear")
    elif h==0:
        print(name1+" is linear")
    elif h<0:
        print(name1+" is sublinear")
    print(h)
    arr=[]
    arr.append(h)
    name1="slopforedgespeed"+",inutrientcon="+str(ccv)+",gamma="+str(gam)+".csv"
    np.savetxt(name1, arr, delimiter=",", fmt="%.6f")
    
    #name1="time,d="+str(D)+",gamma="+str(gam)+".csv"
    #np.savetxt(name1, timerecorded, delimiter=",", fmt="%.6f")
    i=i+1

nutrient density= 1 ,cell capacity= 1 ,ng= 1 ,nk= 0.4 ,gamma= 3
[36.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]
0.0
0.5189873417721519
1.0379746835443038
1.5569620253164556
2.0759493670886076
2.5949367088607596
3.113924050632911


In [6]:
print(1)

1
